# T1.5 확인: 압축 파일 업로드 후 두 경로에서 내려받아 해제 검증

T1.4에서 만든 압축 파일(패치 풀 + 테스트셋)을 공유 Drive와 Kaggle 비공개 Dataset에 올린 뒤, **팀원이 받는 것과 같은 방식**으로 내려받아 데이터가 같은지 확인합니다. 이 노트북은 검증만 하고 로직은 `src/`에 없습니다(저장소 clone도 필요 없습니다).

- 셀 1: 공통 검증 함수 (`content_sha256` 재계산, `patch_pool_id`, 테스트셋 HR/LR 대응)
- 셀 2: **소유자만** — 내 Drive의 원본 zip sha256 계산 (Colab)
- 셀 3: **경로 A (Colab)** — Drive 공유 링크로 받기(마운트 없이) → sha256 비교 → 해제 → 검증
- 셀 4: **경로 B (Kaggle 노트북)** — `/kaggle/input/<slug>/` 검증 (zip이 자동 해제돼 있으면 해제본만 검증)
- 셀 5: 요약 (이 출력을 그대로 붙여 달라고 전달)

주의
- 저장소가 Public입니다. **공유 링크와 Dataset slug는 이 파일·출력·커밋에 남기지 않습니다.** 셀이 `getpass`로만 받습니다.
- 경로 A는 **zip 파일 하나의 공유 링크**(폴더 링크 아님)가 필요합니다. "링크가 있는 모든 사용자 — 뷰어"로 설정하세요.
- 셀 3과 4는 서로 다른 환경입니다. 각각 실행하고 두 요약을 모두 전달합니다.
- 이 풀은 파일럿용입니다(패치 96, 10000개, 시드 42). 본 실행 풀은 T4.2 이후 다시 만들어 같은 방식으로 올립니다.


In [ ]:
import hashlib, json, os, time, zipfile
from pathlib import Path
import numpy as np
from PIL import Image

EXPECTED_PATCH_POOL_ID = "div2k_x4_p96_n10000_s42_e8bc539e"   # T1.4 Colab 확인에서 기록한 값
ARCHIVE_NAME = f"{EXPECTED_PATCH_POOL_ID}.zip"
EXPECTED_TESTSET_COUNTS = {"Set5": 5, "Set14": 14, "BSD100": 100, "Urban100": 100}
SCALE = 4

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

def content_sha256(hr, lr):
    # src/sr_pipeline/data/prepare.py의 _content_sha256과 같은 알고리즘이다(저장소 없이 Kaggle에서도 돌리려고 복제).
    digest = hashlib.sha256()
    for array in (hr, lr):
        digest.update(str(array.shape).encode("ascii"))
        digest.update(np.ascontiguousarray(array))
    return digest.hexdigest()

def find_unpacked_root(directory):
    """patch_pool/metadata.json이 있는 폴더를 찾는다. Kaggle 자동 해제는 한 단계 더 감쌀 수 있다."""
    directory = Path(directory)
    candidates = [directory] + sorted(p.parent.parent for p in directory.rglob("patch_pool/metadata.json"))
    for candidate in candidates:
        if (candidate / "patch_pool" / "metadata.json").is_file():
            return candidate
    raise AssertionError(f"{directory} 아래에서 patch_pool/metadata.json을 찾지 못했습니다")

def verify_archive_dir(directory):
    """해제된 폴더가 T1.4가 만든 산출물과 같은지 검사한다. {검사 이름: bool}을 돌려준다."""
    root = find_unpacked_root(directory)
    pool_dir = root / "patch_pool"
    meta = json.loads((pool_dir / "metadata.json").read_text(encoding="utf-8"))
    hr = np.load(pool_dir / "hr.npy")
    lr = np.load(pool_dir / "lr.npy")
    n, p = meta["num_patches"], meta["patch_size_hr"]

    checks = {}
    checks["hr/lr 배열 dtype·shape가 metadata와 일치"] = (
        hr.dtype == np.uint8 and lr.dtype == np.uint8
        and hr.shape == (n, p, p, 3) and lr.shape == (n, p // SCALE, p // SCALE, 3))
    checks["content_sha256 재계산 일치"] = content_sha256(hr, lr) == meta["content_sha256"]
    checks["patch_pool_id 접미사 = content_sha256[:8]"] = (
        meta["patch_pool_id"] == f"div2k_x4_p{p}_n{n}_s{meta['pool_seed']}_{meta['content_sha256'][:8]}")
    checks[f"patch_pool_id = {EXPECTED_PATCH_POOL_ID}"] = meta["patch_pool_id"] == EXPECTED_PATCH_POOL_ID

    testsets = root / "testsets"
    testsets_meta = json.loads((testsets / "metadata.json").read_text(encoding="utf-8"))
    checks["테스트셋 영상 수 기록"] = testsets_meta["image_counts"] == EXPECTED_TESTSET_COUNTS
    for name, expected in EXPECTED_TESTSET_COUNTS.items():
        hr_files = sorted(f.name for f in (testsets / name / "HR").iterdir())
        lr_files = sorted(f.name for f in (testsets / name / "LR").iterdir())
        names_ok = len(hr_files) == expected and hr_files == lr_files
        sizes_ok = names_ok   # 파일명이 안 맞으면 열어 보지 않고 바로 FAIL로 둔다
        for file_name in (hr_files if names_ok else []):
            with Image.open(testsets / name / "HR" / file_name) as h, Image.open(testsets / name / "LR" / file_name) as l:
                sizes_ok &= (h.width % SCALE == 0 and h.height % SCALE == 0
                             and l.size == (h.width // SCALE, h.height // SCALE))
        checks[f"{name}: {expected}장, HR/LR 파일명 대응·크기(HR/4)"] = names_ok and sizes_ok
    return checks, meta

def report(title, checks):
    for name, ok in checks.items():
        print("OK  " if ok else "FAIL", name)
    passed = all(checks.values())
    print(f"{title}:", "PASS" if passed else "FAIL")
    return passed

RESULTS = {}   # 이 세션(환경)에서 실행한 경로의 결과
print("셀 1 OK")


In [ ]:
# 소유자만 실행(Colab): 내 Drive에 있는 원본 zip의 sha256을 구한다. 이 값이 경로 A·B 비교 기준이다.
import glob
from google.colab import drive
drive.mount("/content/drive")

# 폴더 이름을 코드에 박지 않고 MyDrive 아래에서 파일명으로 찾는다(정확히 1개여야 한다).
matches = glob.glob(f"/content/drive/MyDrive/**/{ARCHIVE_NAME}", recursive=True)
assert len(matches) == 1, (
    f"MyDrive 아래에서 {ARCHIVE_NAME}을 {len(matches)}개 찾았습니다(1개여야 함): {matches}. "
    "0개면 T1.4 노트북 마지막 셀로 먼저 복사하거나, 공유 문서함 폴더는 '내 드라이브에 바로가기 추가'를 하세요.")
OWNER_DRIVE_PATH = matches[0]
print("원본 위치   :", OWNER_DRIVE_PATH)

ORIGINAL_SHA256 = sha256_of(OWNER_DRIVE_PATH)
print("원본 sha256 :", ORIGINAL_SHA256)
print("원본 크기   :", f"{os.path.getsize(OWNER_DRIVE_PATH) / 1e6:.0f} MB")
print("-> 이 sha256은 비밀이 아니다. 셀 3·4에서 비교용으로 입력한다.")


In [ ]:
# 경로 A (Colab): 마운트 없이 공유 링크로 받는다. 팀원이 하는 방식 그대로다.
import getpass, subprocess, sys, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
import gdown

link = getpass.getpass("Drive 공유 링크(zip 파일 하나, 화면에 표시되지 않음): ")
original_sha256 = input("원본 sha256(셀 2 출력): ").strip().lower()
assert len(original_sha256) == 64, "sha256은 64자 16진수입니다"

# 이 두 폴더는 이 셀만 쓰는 임시 폴더다. 이전 실행이 남겼으면 지우고 새로 만든다(Drive 원본은 건드리지 않는다).
DOWNLOAD_DIR = Path("/content/t1_5_download_a")
UNPACK_DIR_A = Path("/content/t1_5_unpacked_a")
for path in (DOWNLOAD_DIR, UNPACK_DIR_A):
    shutil.rmtree(path, ignore_errors=True)
DOWNLOAD_DIR.mkdir(parents=True)

start = time.time()
downloaded = DOWNLOAD_DIR / ARCHIVE_NAME
try:
    try:
        result_path = gdown.download(url=link, output=str(downloaded), fuzzy=True, quiet=True)
    except Exception as error:
        raise RuntimeError(f"gdown 실패({type(error).__name__}). 링크 권한(링크가 있는 모든 사용자)과 파일 링크 여부를 확인하세요.") from None
    finally:
        del link   # 이 세션 메모리에서도 지운다
    assert result_path is not None and downloaded.is_file(), "내려받기 실패: 공유 권한 또는 대용량 파일 확인 페이지 문제일 수 있습니다."
except BaseException:
    shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)   # 내려받기에 실패하면 반쪽 폴더를 남기지 않는다
    raise
elapsed_download = time.time() - start

size_mb = downloaded.stat().st_size / 1e6
sha256_downloaded = sha256_of(downloaded)
checks = {"받은 zip의 sha256 = 원본 sha256": sha256_downloaded == original_sha256}
with zipfile.ZipFile(downloaded) as archive:
    checks["zip 무결성(testzip)"] = archive.testzip() is None
    assert all(checks.values()), "내려받은 파일이 원본과 다릅니다(위 결과 확인). 해제하지 않고 중단합니다."
    archive.extractall(UNPACK_DIR_A)
pool_checks, meta_a = verify_archive_dir(UNPACK_DIR_A)
checks.update(pool_checks)
passed = report("경로 A (Drive 공유 링크)", checks)
RESULTS["A_drive_link"] = {"passed": passed, "sha256": sha256_downloaded, "size_mb": round(size_mb),
                           "download_seconds": round(elapsed_download), "patch_pool_id": meta_a["patch_pool_id"]}
assert passed
print("셀 3 OK")


In [ ]:
# 경로 B (Kaggle 노트북): Dataset을 Add Input으로 붙인 뒤 실행한다. 이 Dataset은 비공개여야 한다.
# Kaggle에서 실행하는 이유: 팀원이 GPU를 못 받았을 때 쓰는 백업 경로가 실제로 되는지 보려는 것이다.
# 먼저 셀 1을 이 Kaggle 노트북에서도 실행해야 한다(Colab과 별개의 커널이다).
import sys
assert "verify_archive_dir" in globals(), "셀 1을 먼저 이 노트북에서 실행하세요(함수·상수 정의)."
assert "google.colab" not in sys.modules, "이 셀은 Colab이 아니라 Kaggle 노트북에서 실행합니다(Colab에는 Kaggle Dataset이 없습니다)."
KAGGLE_INPUT = Path("/kaggle/input")
assert KAGGLE_INPUT.exists(), "경로 B는 Kaggle 노트북에서 실행합니다(/kaggle/input 없음)."

# Dataset 경로가 /kaggle/input/<slug> 일 수도, /kaggle/input/datasets/<유저>/<slug> 일 수도 있어서
# 경로를 입력받지 않고 붙은 입력 전체에서 찾는다. (경로·유저명·slug는 출력하지 않는다.)
zips = sorted(z for z in KAGGLE_INPUT.rglob("*.zip") if z.name == ARCHIVE_NAME)
unpacked_roots = sorted({p.parent.parent for p in KAGGLE_INPUT.rglob("patch_pool/metadata.json")})
assert zips or unpacked_roots, (
    f"/kaggle/input 아래에서 {ARCHIVE_NAME} 도, 해제된 patch_pool/ 도 찾지 못했습니다. "
    "오른쪽 Input 패널의 Add Input으로 Dataset을 붙였는지 확인하세요.")

checks = {}
record = {"zip_preserved": bool(zips)}
if zips:
    assert len(zips) == 1, f"{ARCHIVE_NAME}이 {len(zips)}개 붙어 있습니다: 하나만 붙이세요"
    original_sha256 = input("원본 sha256(셀 2 출력): ").strip().lower()
    assert len(original_sha256) == 64, "sha256은 64자 16진수입니다"
    record["sha256"] = sha256_of(zips[0])
    checks["Kaggle의 zip sha256 = 원본 sha256"] = record["sha256"] == original_sha256
    UNPACK_DIR_B = Path("/kaggle/working/t1_5_unpacked_b")
    assert not UNPACK_DIR_B.exists(), f"{UNPACK_DIR_B}가 이미 있습니다. 지우고 다시 실행하세요."
    with zipfile.ZipFile(zips[0]) as archive:
        checks["zip 무결성(testzip)"] = archive.testzip() is None
        assert all(checks.values()), "Kaggle의 zip이 원본과 다릅니다. 해제하지 않고 중단합니다."
        archive.extractall(UNPACK_DIR_B)
    verify_dir = UNPACK_DIR_B
else:
    assert len(unpacked_roots) == 1, f"해제된 patch_pool/이 {len(unpacked_roots)}곳에 있습니다: Dataset을 하나만 붙이세요"
    print("zip이 남아 있지 않습니다(Kaggle이 자동 해제). 원본 sha256 대신 해제본의 content_sha256 재계산으로 데이터 동일성을 확인합니다.")
    verify_dir = unpacked_roots[0]

pool_checks, meta_b = verify_archive_dir(verify_dir)
checks.update(pool_checks)
passed = report("경로 B (Kaggle 비공개 Dataset)", checks)
record.update({"passed": passed, "patch_pool_id": meta_b["patch_pool_id"]})
RESULTS["B_kaggle_dataset"] = record
assert passed
print("셀 4 OK")


In [ ]:
# 이 출력을 그대로 복사해 전달한다(T1.5 통과 근거). 링크·slug는 출력하지 않는다.
import platform
print("platform :", platform.platform())
print("expected :", EXPECTED_PATCH_POOL_ID)
if not RESULTS:
    print("이 세션에서 실행한 경로가 없습니다(셀 3 또는 셀 4 실행).")
for key, value in RESULTS.items():
    print(f"[{key}]", "PASS" if value["passed"] else "FAIL")
    for field, v in value.items():
        if field != "passed":
            print(f"    {field}: {v}")
